# N156 · SQL日期、留存与移动报表

**目标：** 对齐日期粒度后计算分母和滑窗。

**先修：** N155。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 日期运算；日期缺口；日聚合；首日/次日；分母；四舍五入；执行计划导览。

**配套讲解来源：** [同名逐章意见](../../comment/156_sql_dates_retention_reports.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

上一章（N155）我们把"连续"理解成"id 加一"或"行号"，这一章要把"连续"对齐到**自然日历**上。三类问题：

1. **比昨天热的是哪天**（LeetCode 197）：找出比前一天温度更高的日期。坑在于"前一天"是日历上的昨天，不是表里的上一行。
2. **次日留存率**（LeetCode 550）：每个玩家注册（首次登录）后的**第二天**还登不登录？算一个比例。坑在于分母是"玩家数"不是"登录行数"，而且同一玩家同一天登两次不能算两次留存。
3. **餐厅 7 天滚动营业额**（LeetCode 1321）：对每个有消费的日期，算"它和它前面六个自然日"的消费总和与日均值。坑在于表里有些日期没有消费记录，你不能拿"最近 7 行"冒充"最近 7 天"。

给你一个具体到数字的例子（第 3 题）：表里只有 4 个有消费的日子——1 月 1 日 30 元、1 月 3 日 30 元、1 月 7 日 70 元、1 月 8 日 80 元。问 1 月 8 日那天，"过去 7 天（含当天）"的总额是多少？答案不是 30+30+70+80=210（那是不管日期只数 4 行），而是 **180**：因为 7 天窗口是 1 月 2 日到 1 月 8 日，1 月 1 日已经掉出窗口了，只剩 1 月 3、7、8 三天，30+70+80=180。这一章就是教你把"窗口"定义在**日期数值**上，而不是行数上。

## 2. 基础知识：先读懂这些词

- **日期不是普通数字，但可以变成数字**：SQLite 里日期常写成 `'2021-01-07'` 这种 TEXT。`DATE(x, '-1 day')` 算出 x 的前一天，`JULIANDAY(x)` 把日期换算成"儒略日数"（一个连续递增的整数天数），比如 1 月 8 日比 1 月 1 日恰好大 7。一旦变成数字，就能比较、能相减、能当窗口排序键。
- **自然日连续 vs 相邻行**：上一章的连续是"下一行"，这一章的连续是"日历上差一天"。表里缺了 1 月 2 日的记录，不代表 1 月 3 日不是 1 月 1 日的"两天后"。
- **首日（first_date）**：每个玩家的最早登录日，`MIN(event_date)` 按玩家分组求出。它是留存分析的时间锚点。
- **分母**：算比例时的"总体个数"。留存率的分母是**独立玩家个数**（本章是 3 个玩家），不是登录记录条数（本章是 6 条）。分母选错，比例就整个错了。
- **EXISTS**：一个返回真/假的子查询判断——"存在满足条件的行吗？"它天然去重：玩家 1 在 1 月 1 日登了两条记录，EXISTS 只说"存在"，不会把 1 数成 2。
- **四舍五入 ROUND(x, 2)**：把数值保留两位小数，比如 1/3 = 0.333… 变成 0.33。
- **COALESCE(a, b)**："如果 a 是 NULL 就取 b"。空表时 SUM 返回 NULL，除法也得到 NULL，COALESCE 把它兜底成 0.0。
- **RANGE 窗口 vs ROWS 窗口**：`ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` 数的是**物理行**（往前数 6 行），`RANGE BETWEEN 6 PRECEDING ...` 数的是**排序键的数值差**（排序键往前减 6 以内的行都算）。日期题必须用 RANGE，因为缺日期时"往前 6 行"和"往前 6 天"不是一回事。
- **日聚合（daily CTE）**：先把同一日期的多笔消费 `SUM` 成一天一行，再做窗口计算；否则同一天的行会在窗口里被重复计入。

## 3. 思路推导：从小例子开始

**Step 1：找"比昨天热"的日期。** 表 `Weather` 有三条记录：`(id1, 2020-12-31, 10)`、`(id2, 2021-01-01, 11)`、`(id3, 2021-01-03, 20)`。思路是把表和自己连一次，连接条件不是"id 差一"而是"昨天的 recordDate 恰好等于我的前一天"。手算：id2 的昨天是 2020-12-31，存在，温度 11 > 10 → 入选；id3 的昨天是 2021-01-02，表里**根本没有这一天**，连接失败 → 落选；id1 的昨天 2020-12-30 也不存在 → 落选。最终答案只有 `id=2`。注意陷阱：如果你用上一章的 LAG 取"上一行"，id3 的上一行是 id1（2021-01-01，11 度），20 > 11 会让 id3 被错误地选中——上一行不等于昨天。

**Step 2：次日留存率。** 表 `Activity` 六条记录：玩家 1 在 12-31、01-01（两条）、玩家 2 在 01-01、01-03、玩家 3 在 01-02。先求首日：玩家 1 → 2020-12-31，玩家 2 → 2021-01-01，玩家 3 → 2021-01-02，所以**分母是 3 个玩家**（不是 6 条日志）。再逐个判断"首日+1 天那天有没有登录"，列成手算表：

| 玩家 | 首日 | 首日+1 天 | 当天有登录吗 | 记分 |
|------|------|-----------|--------------|------|
| 1 | 2020-12-31 | 2021-01-01 | 有（两条记录，但"存在"只算一次） | 1 |
| 2 | 2021-01-01 | 2021-01-02 | 没有（他 3 号才来） | 0 |
| 3 | 2021-01-02 | 2021-01-03 | 没有（只来过这一天） | 0 |

分子 = 1，分母 = 3，留存率 = 1/3 = 0.333…，四舍五入两位 = **0.33**。

**Step 3：餐厅 7 天滚动窗口。** 先日聚合：`daily` 表为 `01-01 → 30, 01-03 → 30, 01-07 → 70, 01-08 → 80`。再把日期换算成 JULIANDAY（不妨把 01-01 记作第 0 天，则 01-03=+2、01-07=+6、01-08=+7）。对每一天开 `RANGE BETWEEN 6 PRECEDING AND CURRENT ROW` 的窗口，意思是"数值差在 6 以内的行都进来"。逐天手算：

| 日期 | 窗口覆盖的自然日 | 窗口内的行 | 汇总 amount |
|------|------------------|-----------|-------------|
| 01-01 | 12-26 ~ 01-01 | 01-01 | 30 |
| 01-03 | 12-28 ~ 01-03 | 01-01, 01-03 | 60 |
| 01-07 | 01-01 ~ 01-07 | 01-01, 01-03, 01-07 | 130 |
| 01-08 | 01-02 ~ 01-08 | 01-03, 01-07, 01-08 | **180** |

看 01-08 这一行：01-01 与它差 7 天，被 RANGE 窗口排除在外，所以是 180 而不是 210。这就是"缺失日期不能把七行窗口伪装成七天"的正面演示。最后只输出"从最早有消费的日子算起至少过了 6 天"的日期（01-07、01-08），因为更早的日期凑不满完整 7 天窗口，这是题目要求；输出时总额除以固定分母 7.0 再保留两位小数。

## 4. 核心代码：query_rising_temperature.sql

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```sql
SELECT today.id
FROM Weather AS today JOIN Weather AS yesterday
ON yesterday.recordDate = DATE(today.recordDate, '-1 day')
WHERE today.temperature > yesterday.temperature;
```

### 逐段读懂代码

“分段实现”部分 仍然是"Python 字典装 SQL + 内存 SQLite 夹具"的结构，我们按三条 SQL 逐条看。

**第 1 条：比昨天热（对应 Step 1）。**

```sql
SELECT today.id
FROM Weather AS today JOIN Weather AS yesterday
ON yesterday.recordDate = DATE(today.recordDate, '-1 day')
WHERE today.temperature > yesterday.temperature;
```

同一张 `Weather` 起两个别名：`today` 是候选日，`yesterday` 是它的昨天。`DATE(today.recordDate, '-1 day')` 用 SQLite 的日期修饰符算出前一天；JOIN 条件要求表里**真的存在**这一天的记录，缺日期自动落选。WHERE 再比温度。

**第 2 条：次日留存率（对应 Step 2）。**

```sql
WITH first_login AS (
    SELECT player_id, MIN(event_date) AS first_date FROM Activity GROUP BY player_id
)
SELECT ROUND(COALESCE(1.0 * SUM(EXISTS (
    SELECT 1 FROM Activity AS a
    WHERE a.player_id = f.player_id AND a.event_date = DATE(f.first_date, '+1 day')
)) / COUNT(*), 0.0), 2) AS fraction
FROM first_login AS f;
```

CTE `first_login` 按玩家分组取 `MIN(event_date)` 当首日——分母从此变成"一行一个玩家"。外层对 `first_login` 的每一行跑一次 `EXISTS` 子查询：在 Activity 里找"该玩家、首日+1 天"的记录，找到记 1，找不到记 0；`SUM` 把这些 0/1 加起来就是分子。`1.0 *` 是故意把整数变成浮点，避免 1/3 在整数除法下被截成 0；`COUNT(*)` 是玩家数分母；`COALESCE(..., 0.0)` 兜住空表（空表时 SUM 是 NULL，SQLite 里除零也得 NULL，都换回 0.0）；最后 `ROUND(..., 2)` 保留两位。

**第 3 条：餐厅 7 天滚动（对应 Step 3）。**

（对应程序见下方分段实现与完整代码。）

分三层：`daily` 把同一日期的多笔消费压成一天一行（否则同一天三笔会被窗口当三天重复加）；`rolling` 用 `SUM(...) OVER (ORDER BY JULIANDAY(visited_on) RANGE BETWEEN 6 PRECEDING AND CURRENT ROW)` 做滚动求和——排序键是儒略日**数值**，RANGE 按数值差 6 圈定窗口，这正是"7 个自然日"的准确表达；最外层 `WHERE JULIANDAY(visited_on) - (首日儒略日) >= 6` 只留"满窗"的日期，均值固定除以 7.0（不是窗口里实际有行的天数），并按日期排序输出。

**夹具部分**（`create_sqlite_fixture` / `query_rows`）和 N155 相同套路：`:memory:` 内存库建 `Weather`、`Activity`、`Customer` 三张表灌小数据，`query_rows` 按名字执行查询并返回元组列表，供 assert 用。

另外，“运行示例”部分 会把三条查询的结果各渲染成一张 HTML 表（`show_table` 辅助函数负责把表头和行拼成表格），并打印一行 `实际SQLite版本 3.45.1`——这行输出是在提醒你：本章所有日期函数行为都以这个真实引擎版本为准，跑之前先预测输出，再对照这批表检查。

表里值得核对的两处：`query_rising_temperature.sql` 的结果表只有一行 `(2,)`；`query_restaurant_growth.sql` 的结果表恰好两行（01-07 与 01-08），和第三节的手算逐一对应。

## 5. 分段实现：按顺序运行

**本章接口：** `query_rising_temperature.sql`、`query_next_day_retention.sql`、`query_restaurant_growth.sql`

### 导入本章使用的库

In [1]:
import sqlite3

### SQL_QUERIES

In [2]:
SQL_QUERIES = {
    'query_rising_temperature.sql': r'''SELECT today.id
FROM Weather AS today JOIN Weather AS yesterday
ON yesterday.recordDate = DATE(today.recordDate, '-1 day')
WHERE today.temperature > yesterday.temperature;''',
    'query_next_day_retention.sql': r'''WITH first_login AS (
    SELECT player_id, MIN(event_date) AS first_date FROM Activity GROUP BY player_id
)
SELECT ROUND(COALESCE(1.0 * SUM(EXISTS (
    SELECT 1 FROM Activity AS a
    WHERE a.player_id = f.player_id AND a.event_date = DATE(f.first_date, '+1 day')
)) / COUNT(*), 0.0), 2) AS fraction
FROM first_login AS f;''',
    'query_restaurant_growth.sql': r'''WITH daily AS (
    SELECT visited_on, SUM(amount) AS daily_amount FROM Customer GROUP BY visited_on
), rolling AS (
    SELECT visited_on,
           SUM(daily_amount) OVER (
               ORDER BY JULIANDAY(visited_on)
               RANGE BETWEEN 6 PRECEDING AND CURRENT ROW
           ) AS amount
    FROM daily
)
SELECT visited_on, amount, ROUND(amount / 7.0, 2) AS average_amount
FROM rolling
WHERE JULIANDAY(visited_on) - (SELECT JULIANDAY(MIN(visited_on)) FROM daily) >= 6
ORDER BY visited_on;''',
}

### create_sqlite_fixture

In [3]:
def create_sqlite_fixture():
    connection = sqlite3.connect(':memory:')
    connection.executescript("\nCREATE TABLE Weather(id INTEGER PRIMARY KEY,recordDate TEXT,temperature INTEGER);\nINSERT INTO Weather VALUES (1,'2020-12-31',10),(2,'2021-01-01',11),(3,'2021-01-03',20);\nCREATE TABLE Activity(player_id INTEGER,device_id INTEGER,event_date TEXT,games_played INTEGER);\nINSERT INTO Activity VALUES (1,1,'2020-12-31',1),(1,1,'2021-01-01',1),(1,2,'2021-01-01',2),\n(2,1,'2021-01-01',1),(2,1,'2021-01-03',1),(3,1,'2021-01-02',1);\nCREATE TABLE Customer(customer_id INTEGER,name TEXT,visited_on TEXT,amount INTEGER);\nINSERT INTO Customer VALUES (1,'A','2021-01-01',10),(2,'B','2021-01-01',20),\n(3,'C','2021-01-03',30),(4,'D','2021-01-07',70),(5,'E','2021-01-08',80);\n")
    return connection

### query_rows

In [4]:
def query_rows(connection, name):
    return connection.execute(SQL_QUERIES[name]).fetchall()

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

db=create_sqlite_fixture()
for name in SQL_QUERIES:
    cursor=db.execute(SQL_QUERIES[name]); print(name); show_table([x[0] for x in cursor.description],cursor.fetchall())
print('实际SQLite版本',sqlite3.sqlite_version)
db.close()

query_rising_temperature.sql


id
2


query_next_day_retention.sql


fraction
0.33


query_restaurant_growth.sql


visited_on,amount,average_amount
2021-01-07,130,18.57
2021-01-08,180,25.71


实际SQLite版本 3.45.1


## 7. 正确性、适用条件与复杂度

留存先按玩家求首日，再用EXISTS判断恰好次日是否出现，重复登录不重复计分。餐厅RANGE以JULIANDAY数值排序，6 PRECEDING到CURRENT ROW对应当前日及前六个自然日；与ROWS基于行数不同。分母固定7，空活动集合的0.0是本章明确的教学扩展。

**代价：** 先聚合日/玩家通常需分组与排序；窗口排序O(D log D)级，实际查询计划决定具体实现。DATE、JULIANDAY及此窗口写法已在SQLite执行，未宣称MySQL/PostgreSQL方言已测。

### 展开理解

**留存为什么对？** 关键在两处。第一，分母用的是 `first_login`（按玩家 MIN 出首日，一人一行），所以不管玩家 1 在 1 月 1 日插多少条记录，分母都是 3 不动。第二，分子用的是 `EXISTS` 而不是 `COUNT(记录数)`，它只回答"次日到底来没来"，来十次也只记 1 分。这两个选择合起来保证了"重复登录不重复计分"。

**餐厅为什么对？** 因为窗口定义在 JULIANDAY 的**数值差**上。RANGE 窗口的语义是"排序键的值落在 `[当前值-6, 当前值]` 区间内的行"，换算回日历正好是 7 个自然日；缺失的日期没有行，自然贡献 0，也绝不会把窗口"撑大"或"缩小"。如果我们用 ROWS 6 PRECEDING，窗口就变成"物理上前 6 行"，日期一缺，7 行可能横跨十几个自然日，总额就错了。`>= 6` 那行保证输出的每个日期都有完整的 7 天历史可查。空活动集合输出 0.0、空 Customer 输出空列表，这两条是本章明确的教学扩展行为（原题不一定有空表输入）。

**复杂度怎么说？** 三条查询都先要"分组"或"排序"：留存先按玩家分组再对每个玩家做一次 EXISTS 子查询（走索引的话每次接近常数，整体约 O(n) 到 O(n log n)）；餐厅先按日期分组 O(n log n)，窗口再按天数 D 排序 O(D log D)。拿本章的量级举例：5 笔 Customer 记录、4 个不同日期，log 4 = 2，代价小到可以忽略；就算放大到一年的流水 10 万行、365 个日聚合行，窗口排序也只有 365 × log 365 ≈ 3000 次比较这个量级。再具体一点：RANGE 滑窗扫一遍聚合行是线性的，365 行就 365 步，每步圈定"数值差 ≤ 6"的邻居——已经按儒略日排好序，邻居在物理上也挨着，不用回头全表找。

还有一个容易忽略的量级差异值得单独说：日聚合把"10 万行流水"压成"365 个日聚合行"之后再做窗口，窗口部分只跟 365 有关；如果不聚合直接在 10 万行上开窗，同样的窗口语义要处理 10 万行的排序和圈定，差了近三个数量级。这也是 “思路推导”部分 强调"先按日期汇总，再做窗口"的原因之一：先做聚合（统计单位对了），后面每一步都便宜。

最后提醒：`DATE`、`JULIANDAY`、RANGE 窗口这些写法是在 SQLite（“运行示例”部分 打印出实际版本 3.45.1）里真实跑通的，换成 MySQL/PostgreSQL 方言要改写（练习 2 就让你做这件事）。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 先重建数据库，然后逐条 assert，最后还做了三个"改表再测"的突变验证。跑测试前，你可以先把三条查询的预期输出默写出来当"考前预测"：

- `query_rising_temperature.sql` → 一行 `(2,)`（只有 1 月 1 日比昨天热）
- `query_next_day_retention.sql` → 一行 `(0.33,)`（3 个玩家 1 个次日回来）
- `query_restaurant_growth.sql` → 两行 `(2021-01-07, 130, 18.57)` 和 `(2021-01-08, 180, 25.71)`
- 写完再对照 assert，四个预言全中说明第三节的推导你已经吃透了。

1. `assert query_rows(db,'query_rising_temperature.sql')==[(2,)]` —— 正常值 + 日期缺口：id2（比昨天热）入选；id3 虽然 20 度最高，但 1 月 2 日无记录、它没有"昨天"可比，必须落选。这条就是在防"用上一行冒充昨天"的错误写法。
2. `assert query_rows(db,'query_next_day_retention.sql')==[(0.33,)]` —— 正常值：3 个玩家、1 个次日回来，1/3 四舍五入到两位是 0.33。同时隐含测了玩家 1 同日两条登录没有把分子撑成 2、分母撑成 4。
3. `assert query_rows(db,'query_restaurant_growth.sql')==[('2021-01-07',130,18.57),('2021-01-08',180,25.71)]` —— 正常值 + 满窗边界：01-07 是第一个凑满 7 天窗口的日期（130 = 30+30+70，均值 130/7≈18.57）；01-08 验证 01-01 已滑出窗口（180 = 30+70+80，均值 180/7≈25.71）。01-01 和 01-03 不满 6 天差，不许出现在结果里。
4. `INSERT INTO Activity VALUES(1,3,'2021-01-01',99)` 后再 assert 留存仍是 0.33 —— **重复值测试**：给玩家 1 再塞一条同日登录，验证"分母是玩家数、EXISTS 只记一次"的契约。
5. `DELETE FROM Activity` 后 assert 留存是 `[(0.0,)]` —— **空表边界**：没有玩家时 SUM/COUNT 都落空，COALESCE 必须把 NULL 兜成 0.0 而不是报错或返回空。
6. `DELETE FROM Customer` 后 assert 餐厅查询是 `[]` —— **空表边界**：没有消费就没有日聚合、没有满窗日期，输出必须是空列表而不是崩溃。

In [6]:
db = create_sqlite_fixture()

assert query_rows(db, 'query_rising_temperature.sql') == [(2,)]

assert query_rows(db, 'query_next_day_retention.sql') == [(0.33,)]

assert query_rows(db, 'query_restaurant_growth.sql') == [('2021-01-07', 130, 18.57), ('2021-01-08', 180, 25.71)]

db.execute("INSERT INTO Activity VALUES(1,3,'2021-01-01',99)")

assert query_rows(db, 'query_next_day_retention.sql') == [(0.33,)]

db.execute('DELETE FROM Activity')

assert query_rows(db, 'query_next_day_retention.sql') == [(0.0,)]

db.execute('DELETE FROM Customer')

assert query_rows(db, 'query_restaurant_growth.sql') == []

db.close()

print('N156: 所有本章断言通过')

N156: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 说明上一条记录不一定是昨天。

**练习 2：** 比较SQLite教学SQL和题目MySQL方言，分别标注。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（说明"上一条记录不一定是昨天"）**：提示——就用本章 Weather 的三行数据：1 月 2 日没有记录，所以 1 月 3 日在表里的上一行是 1 月 1 日，中间隔了两天。先手算"LAG 上一行版"会把 id3 选进来（20 > 11），而"DATE 前一天版"不会；再分别写出两种 SQL，用 `query_rising_temperature.sql` 的夹具跑，对比结果差在哪一行。边界可以再加一条"两天前的记录温度更高"的数据，看两个版本给出的答案如何分叉。
- **练习 2（对比 SQLite 教学写法和 MySQL 方言）**：提示——列一个两栏对照表：SQLite 的 `DATE(x, '-1 day')` 在 MySQL 里是 `DATE_SUB(x, INTERVAL 1 DAY)`；`JULIANDAY(x)` 在 MySQL 里常用 `DATEDIFF(x, y)` 或 `TO_DAYS(x)`；`ROUND` 两者都有但空值兜底 MySQL 可用 `IFNULL`。逐条标注本章三条 SQL 的哪一处需要替换，替换后语义是否完全一致（特别是 RANGE 窗口在旧版 MySQL 里不支持，可能要改成自连接按日期范围 JOIN）。先用本章数据手算一遍预期输出不变，再验证。建议按三步交作业：

- 表格：每个函数一行，左边 SQLite 写法、右边 MySQL 等价写法。
- 手算：三条查询各自跑前的预期输出（见第六节开头的预测清单）。
- 验证：替换后实际输出与预测逐项对比，指出哪几处语义等价、哪几处有坑。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

本章包含真实的 SQL／Shell／JavaScript／线程程序，Python 部分负责创建示例、调用运行时和断言。嵌入的程序源码也在下面完整显示；缺少相应运行时应安装依赖，不能用模拟结果替代。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
import sqlite3

SQL_QUERIES = {
    'query_rising_temperature.sql': r'''SELECT today.id
FROM Weather AS today JOIN Weather AS yesterday
ON yesterday.recordDate = DATE(today.recordDate, '-1 day')
WHERE today.temperature > yesterday.temperature;''',
    'query_next_day_retention.sql': r'''WITH first_login AS (
    SELECT player_id, MIN(event_date) AS first_date FROM Activity GROUP BY player_id
)
SELECT ROUND(COALESCE(1.0 * SUM(EXISTS (
    SELECT 1 FROM Activity AS a
    WHERE a.player_id = f.player_id AND a.event_date = DATE(f.first_date, '+1 day')
)) / COUNT(*), 0.0), 2) AS fraction
FROM first_login AS f;''',
    'query_restaurant_growth.sql': r'''WITH daily AS (
    SELECT visited_on, SUM(amount) AS daily_amount FROM Customer GROUP BY visited_on
), rolling AS (
    SELECT visited_on,
           SUM(daily_amount) OVER (
               ORDER BY JULIANDAY(visited_on)
               RANGE BETWEEN 6 PRECEDING AND CURRENT ROW
           ) AS amount
    FROM daily
)
SELECT visited_on, amount, ROUND(amount / 7.0, 2) AS average_amount
FROM rolling
WHERE JULIANDAY(visited_on) - (SELECT JULIANDAY(MIN(visited_on)) FROM daily) >= 6
ORDER BY visited_on;''',
}

def create_sqlite_fixture():
    connection = sqlite3.connect(':memory:')
    connection.executescript("\nCREATE TABLE Weather(id INTEGER PRIMARY KEY,recordDate TEXT,temperature INTEGER);\nINSERT INTO Weather VALUES (1,'2020-12-31',10),(2,'2021-01-01',11),(3,'2021-01-03',20);\nCREATE TABLE Activity(player_id INTEGER,device_id INTEGER,event_date TEXT,games_played INTEGER);\nINSERT INTO Activity VALUES (1,1,'2020-12-31',1),(1,1,'2021-01-01',1),(1,2,'2021-01-01',2),\n(2,1,'2021-01-01',1),(2,1,'2021-01-03',1),(3,1,'2021-01-02',1);\nCREATE TABLE Customer(customer_id INTEGER,name TEXT,visited_on TEXT,amount INTEGER);\nINSERT INTO Customer VALUES (1,'A','2021-01-01',10),(2,'B','2021-01-01',20),\n(3,'C','2021-01-03',30),(4,'D','2021-01-07',70),(5,'E','2021-01-08',80);\n")
    return connection

def query_rows(connection, name):
    return connection.execute(SQL_QUERIES[name]).fetchall()

# 示例与回归测试
db = create_sqlite_fixture()

assert query_rows(db, 'query_rising_temperature.sql') == [(2,)]

assert query_rows(db, 'query_next_day_retention.sql') == [(0.33,)]

assert query_rows(db, 'query_restaurant_growth.sql') == [('2021-01-07', 130, 18.57), ('2021-01-08', 180, 25.71)]

db.execute("INSERT INTO Activity VALUES(1,3,'2021-01-01',99)")

assert query_rows(db, 'query_next_day_retention.sql') == [(0.33,)]

db.execute('DELETE FROM Activity')

assert query_rows(db, 'query_next_day_retention.sql') == [(0.0,)]

db.execute('DELETE FROM Customer')

assert query_rows(db, 'query_restaurant_growth.sql') == []

db.close()

print('N156: 所有本章断言通过')

N156: 所有本章断言通过


## 11. 代表题与迁移

- **197. Rising Temperature**：练"日期差一天"的自连接判定，对应 `query_rising_temperature.sql`，专治"把上一行当昨天"的错觉。
- **550. Game Play Analysis IV**：练首日锚点 + EXISTS 判次日 + 正确分母，对应 `query_next_day_retention.sql`。
- **1321. Restaurant Growth**：练日聚合 + RANGE 数值窗口滚动求和，对应 `query_restaurant_growth.sql`，是本章"窗口按天不按行"的核心战场。
- **262. Trips and Users**（extension）：同属"日期粒度 + 报表口径"家族的延伸题，练的是把本章的分母/去重/四舍五入口径意识迁移到禁单率统计上。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。